# 05. 出力と再実行

04章で作ったサマリを**ファイルに書き出し、何度流しても壊れないようにする**のがこの章です。

パイプラインは一度動けば終わり、ではありません。

- 途中で失敗してリトライされる
- 上流のデータが間違っていて、流し直す
- 集計のバグが見つかって、過去分を作り直す
- 誰かが手で叩く

**どれも普通に起きます。** 「一度しか流さない」前提で書いたものは、早い段階で行き詰まります。

## この章のゴール

```
  サマリ (04章)
     ↓  型ごと保存する          Parquet
     ↓  出口で型を強制する       スキーマ (データ契約)
     ↓  月ごとに分けて置く       パーティション
     ↓  差し替えで書く          何度流しても同じ  ← ここがこの章の主題
  out/daily_shop/month=2024-04/part.parquet
  out/daily_category/month=2024-04/part.parquet
```

そのうえで、実際に**流し直します**。

- 02章で隔離した2行の**訂正が届く** → 4月を作り直す
- **5月ぶんが届く** → 4月には触らずに5月だけ足す

## 冪等性

この章のキーワードは1つだけです。

> **冪等(べきとう) = 同じ入力に対して何度実行しても、結果が同じになること。**

読み物 `docs/04 冪等性とリプレイ` に、考え方がまとまっています。
この章は、それを手で確かめる回です。

---
## 0. 前章までのまとめ

**次のセルは01〜04章の答えです。読み飛ばして実行してかまいません。**

In [ ]:
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

# ---------------- 01章: 取り込み ----------------
COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount",
           "tax_type", "note", "source"]

RENAME_STORE = {"売上日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
                "数量": "qty", "金額": "amount", "備考": "note"}
RENAME_EC = {"受注日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
             "数量": "qty", "金額税抜": "amount", "ステータス": "note"}

SOURCES = [
    {"path": "/data/sales_2024-04_old.csv", "encoding": "cp932",
     "tax_type": "税込", "source": "old", "rename": RENAME_STORE},
    {"path": "/data/sales_2024-04_new.csv", "encoding": "utf-8",
     "tax_type": "税込", "source": "new", "rename": RENAME_STORE},
    {"path": "/data/sales_2024-04_ec.csv", "encoding": "utf-8",
     "tax_type": "税抜", "source": "ec", "rename": RENAME_EC},
]


def read_one(spec):
    df = pd.read_csv(spec["path"], dtype=str, keep_default_na=False,
                     encoding=spec["encoding"])
    df = df.rename(columns=spec["rename"])
    df = df.assign(tax_type=spec["tax_type"], source=spec["source"])
    return df[COLUMNS]


def load_raw(sources=SOURCES):
    df = pd.concat([read_one(s) for s in sources], ignore_index=True)
    print(f"取り込み: {len(df)}行  内訳 {df['source'].value_counts().to_dict()}")
    return df


# ---------------- 02章: クレンジング ----------------
NA_TOKENS = ["", "-", "N/A"]
TEXT_COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount", "note"]
FORMATS = ["%Y/%m/%d", "%Y年%m月%d日", "%Y-%m-%d"]
TAX_RATE = 1.1


def norm(s):
    """NFKC正規化して、前後の空白を落とす。"""
    return unicodedata.normalize("NFKC", s).strip()


def parse_date(s):
    d = pd.to_datetime(s, format=FORMATS[0], errors="coerce")
    for fmt in FORMATS[1:]:
        d = d.fillna(pd.to_datetime(s, format=fmt, errors="coerce"))
    return d


def clean_raw(raw):
    """raw を整形して (clean, rejected) に分ける。行は1つも捨てない。"""
    df = raw.copy()
    for c in TEXT_COLUMNS:
        df[c] = df[c].map(norm)
    df = df.replace(NA_TOKENS, pd.NA)
    df["item_cd"] = df["item_cd"].str.zfill(4)
    df["amount"] = df["amount"].str.replace(r"[¥,]", "", regex=True)
    df["sale_date"] = parse_date(df["sale_date"])
    df["qty"] = pd.to_numeric(df["qty"], errors="coerce").astype("Int64")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce").astype("Int64")
    is_excl = df["tax_type"] == "税抜"
    df["amount_incl"] = df["amount"].where(
        ~is_excl, (df["amount"] * TAX_RATE).round()).astype("Int64")
    ok = df["sale_date"].notna() & df["qty"].notna() & df["amount"].notna()
    clean = df[ok].reset_index(drop=True)
    rejected = df[~ok].reset_index(drop=True)
    rate = len(rejected) / len(df) * 100
    print(f"クレンジング: {len(df)}行 → clean {len(clean)}行 / "
          f"rejected {len(rejected)}行 ({rate:.1f}%)")
    return clean, rejected


# ---------------- 03章: 名寄せと結合 ----------------
def load_masters():
    alias = pd.read_csv("/data/shop_alias.csv", dtype=str, keep_default_na=False)
    alias["alias"] = alias["alias"].map(norm)

    shops = pd.read_csv("/data/shops.csv", dtype=str, keep_default_na=False)
    shops = shops.replace("", pd.NA)
    shops["close_date"] = pd.to_datetime(shops["close_date"])

    items = pd.read_csv("/data/items.csv", dtype=str, keep_default_na=False)
    items = items.replace("", pd.NA)
    items["discontinued_date"] = pd.to_datetime(items["discontinued_date"])
    return alias, shops, items


def join_master(clean):
    """clean にマスタを結合する。行は増やさない。除外もしない。"""
    alias, shops, items = load_masters()

    df = clean.rename(columns={"shop_name": "shop_name_raw"})

    df = df.merge(alias, left_on="shop_name_raw", right_on="alias",
                  how="left", validate="m:1").drop(columns="alias")
    df = df.merge(shops[["shop_cd", "shop_name", "area", "close_date"]],
                  on="shop_cd", how="left", validate="m:1")
    df = df.merge(items[["item_cd", "item_name", "category", "discontinued_date"]],
                  on="item_cd", how="left", validate="m:1")

    df["category"] = df["category"].fillna("未分類")

    assert len(df) == len(clean), "結合で行数が変わりました"

    no_shop = df["shop_cd"].isna().sum()
    no_item = df["item_name"].isna().sum()
    closed = (df["close_date"].notna() & (df["sale_date"] > df["close_date"])).sum()
    print(f"結合: {len(df)}行  店舗未マッチ {no_shop} / 商品未マッチ {no_item} / "
          f"閉店後の売上 {closed}")
    return df


# ---------------- 04章: 集計 ----------------
AGG = {"qty": ("qty", "sum"),
       "amount_jpy": ("amount_incl", "sum"),
       "n_lines": ("item_cd", "size")}


def open_shop_cds():
    """営業中の店舗コード。閉店した店は格子に含めない。"""
    _, shops, _ = load_masters()
    return sorted(shops.loc[shops["close_date"].isna(), "shop_cd"])


def select_target(fact):
    """fact から集計対象外を除いて target にする。除外は必ず数える。"""
    is_test = fact["note"].fillna("") == "テスト"
    is_closed = fact["close_date"].notna() & (fact["sale_date"] > fact["close_date"])
    drop = is_test | is_closed

    print(f"除外: テスト伝票 {is_test.sum()}行 "
          f"({fact.loc[is_test, 'amount_incl'].sum():,}円) / "
          f"閉店後 {is_closed.sum()}行 "
          f"({fact.loc[is_closed, 'amount_incl'].sum():,}円)")
    target = fact[~drop].reset_index(drop=True)
    print(f"集計対象: fact {len(fact)}行 → target {len(target)}行  "
          f"{target['amount_incl'].sum():,}円")
    return target


def summarize(fact):
    """fact (除外前) を受け取り、(店舗別, カテゴリ別) の日次サマリを返す。

    除外はこの関数の中でやる。渡すのは fact であって target ではない。
    """
    target = select_target(fact)
    day = target["sale_date"].dt.date.rename("sale_date")

    by_category = (target.groupby([day, "shop_cd", "category"])
                   .agg(**AGG).reset_index())

    # 売上ゼロの日を埋める。埋めてよいのはデータが揃っている範囲だけ
    grid = pd.MultiIndex.from_product(
        [pd.date_range(day.min(), day.max(), freq="D").date, open_shop_cds()],
        names=["sale_date", "shop_cd"])
    by_shop = (target.groupby([day, "shop_cd"])
               .agg(**AGG).reindex(grid, fill_value=0).reset_index())

    total = target["amount_incl"].sum()
    assert by_category["amount_jpy"].sum() == total, "カテゴリ別の合計が target と合いません"
    assert by_shop["amount_jpy"].sum() == total, "店舗別の合計が target と合いません"

    print(f"サマリ: 店舗別 {len(by_shop)}行 "
          f"(うち売上ゼロ {(by_shop['amount_jpy'] == 0).sum()}行) / "
          f"カテゴリ別 {len(by_category)}行  合計 {total:,}円")
    return by_shop, by_category


import os
import shutil
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq

raw = load_raw()
clean, rejected = clean_raw(raw)
fact = join_master(clean)
by_shop, by_category = summarize(fact)
by_shop.head(3)

---
## 1. なぜ CSV ではなく Parquet か

書き出す先を決めます。まず **CSV に書いて読み戻すと何が起きるか**を見ます。

> 次のセルの先頭で `out/` を消しています。
> この章は同じ場所に何度も書くので、**上から流し直したときに前回の結果が残っていると、
> 途中の数字が合わなくなる**からです。
> 消されて困るものは `out/` に置かないでください。

In [ ]:
OUT = Path("/work/out")

# この章は out/ を何度も作り変えます。上から流し直したときに前回の結果が
# 混ざらないよう、ここでまっさらにしておきます
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)

by_shop.to_csv(OUT / "sample.csv", index=False)
back_csv = pd.read_csv(OUT / "sample.csv")

by_shop.to_parquet(OUT / "sample.parquet", index=False)
back_pq = pd.read_parquet(OUT / "sample.parquet")

print("書く前   :", type(by_shop["sale_date"].iloc[0]).__name__)
print("CSV往復後 :", type(back_csv["sale_date"].iloc[0]).__name__)
print("Parquet後 :", type(back_pq["sale_date"].iloc[0]).__name__)

```
書く前    : date
CSV往復後  : str        ← 日付ではなくなった
Parquet後 : date       ← 日付のまま
```

**CSV は型を持てません。** 中身はただのテキストなので、
読み戻した側がもう一度型を推測することになります。
01章でやった「文字コード」「日付の書式」「先頭ゼロ」の苦労を、
**下流にもう一度やらせる**ということです。

Parquet は列ごとに型を持って保存します。だから読み戻しても同じ型で出てきます。

| | CSV | Parquet |
| --- | --- | --- |
| 型 | 持てない | **列ごとに持つ** |
| 大きさ | そのまま | 圧縮される |
| 列だけ読む | できない(全部読む) | **必要な列だけ読める** |
| 人が開く | できる | 専用のツールが要る |

**最後の行だけが CSV の利点です。** それも「人に渡す用に1枚だけ CSV で出す」
という形にすれば済みます。

> パイプラインの**途中**にあるファイルは、人が読むものではありません。
> 次の処理が読むものです。だから**型が保てる形式**を選びます。
> なぜ列指向が速いのかは、この教材の範囲を超えるので触れません
> (`quest-02 columnar` で扱う予定のテーマです)。

In [ ]:
# ✍ 書いてみる: CSVとParquetのファイルサイズをバイト数で比べてください。
#              (ヒント: Path(...).stat().st_size)

ans = ...   # ここに書く

assert isinstance(ans, tuple) and len(ans) == 2, "(CSVのバイト数, Parquetのバイト数) を返してください"
assert ans[0] > 0 and ans[1] > 0
print("OK")
print(f"CSV {ans[0]:,} バイト / Parquet {ans[1]:,} バイト")

<details>
<summary>答え</summary>

```python
ans = ((OUT / "sample.csv").stat().st_size,
       (OUT / "sample.parquet").stat().st_size)
```

</details>

30行しかないので、**Parquet のほうが大きい**はずです。
Parquet はファイルの末尾に型やスキーマの情報を持つので、小さいデータでは負けます。

**行数が増えると逆転します。** 数万行を超えたあたりから、圧縮が効いて差が開きます。
「Parquet のほうが小さい」は、**大きいデータでの話**だと覚えておいてください。

---
## 2. 出口で型を強制する

Parquet は型を持てますが、**書くときの型がそのまま入る**だけです。
`qty` が文字列のまま入っていれば、文字列として保存されます。

だから出口に**関所**を置きます。「この形でなければ書かせない」という宣言です。
これが**データ契約(data contract)のいちばん小さな形**です。

In [ ]:
SCHEMA_SHOP = pa.schema([
    ("sale_date", pa.date32()),
    ("shop_cd", pa.string()),
    ("qty", pa.int32()),
    ("amount_jpy", pa.int64()),
    ("n_lines", pa.int32()),
])

SCHEMA_CATEGORY = pa.schema([
    ("sale_date", pa.date32()),
    ("shop_cd", pa.string()),
    ("category", pa.string()),
    ("qty", pa.int32()),
    ("amount_jpy", pa.int64()),
    ("n_lines", pa.int32()),
])

table = pa.Table.from_pandas(by_shop, schema=SCHEMA_SHOP, preserve_index=False)
print(table.schema)

通りました。`by_shop` の `qty` は `Int64` でしたが、**`int32` に変換されて**います。

`amount_jpy` を `int64` にしているのは、金額だからです。
`int32` の上限は約21億で、**円単位の売上なら普通に超えます**。
数量は `int32` で十分です。

### 型が合わないと、書く前に止まります

In [ ]:
broken = by_shop.copy()
broken["amount_jpy"] = broken["amount_jpy"].astype(str)   # 金額が文字列になってしまった

try:
    pa.Table.from_pandas(broken, schema=SCHEMA_SHOP, preserve_index=False)
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:120]}")

print()

missing = by_shop.drop(columns="n_lines")                  # 列を1つ落としてしまった
try:
    pa.Table.from_pandas(missing, schema=SCHEMA_SHOP, preserve_index=False)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

**どちらも例外で止まりました。**

これが関所を置く目的です。読み物 `docs/03` の言い方を借りると、

> 出力の直前でスキーマを強制すると、
> **取りこぼした汚れが「黙る障害」ではなく「止まる障害」になります。**

止まる障害は、その場で気づけます。
黙る障害は、3か月後に「この数字おかしくない?」と言われるまで気づけません。
**同じバグでも、後者のほうが何倍も高くつきます。**

スキーマが守ってくれるのは3つです。

| | |
| --- | --- |
| 列の名前 | 綴りを変えたら止まる |
| 列の型 | 数値のはずが文字列になっていたら止まる |
| 列の有無 | 増やしても減らしても止まる |

> `preserve_index=False` を付けているのは、
> **pandas の index を勝手な列として書き出さないため**です。
> これが入ると、スキーマに無い列が増えて意味が分からなくなります。

In [ ]:
# ✍ 書いてみる: by_category を SCHEMA_CATEGORY で Table にして、行数を ans に入れてください。

ans = ...   # ここに書く

assert ans == 28, f"28行のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = pa.Table.from_pandas(by_category, schema=SCHEMA_CATEGORY,
                           preserve_index=False).num_rows
```

</details>

---
## 3. 月ごとに分けて置く

置き場所を決めます。**1つのファイルに全部入れる**か、**分けて置く**かです。

```
out/daily_shop.parquet                          全部1ファイル
out/daily_shop/month=2024-04/part.parquet       月ごとに分ける
out/daily_shop/month=2024-05/part.parquet
```

分ける単位のことを**パーティション**と呼びます。

**分ける単位は「流し直す単位」で決めます。** ここがいちばん大事な考え方です。

- 4月のデータを作り直したい → **4月だけ差し替えられる形**にしておく
- 全部1ファイルだと、4月を直すのに5月も6月も書き直すことになる

このパイプラインは月単位で流すので、月で分けます。

> `month=2024-04` という**ディレクトリ名に `=` を使う書き方**は、
> Hive パーティションと呼ばれる慣習です。
> `pd.read_parquet` や DuckDB や Spark が、これを見て
> **`month` という列があるかのように扱ってくれます。**

In [ ]:
def partition_path(table_dir, month):
    return Path(table_dir) / f"month={month}"


path = partition_path(OUT / "daily_shop", "2024-04")
path.mkdir(parents=True, exist_ok=True)

pq.write_table(
    pa.Table.from_pandas(by_shop, schema=SCHEMA_SHOP, preserve_index=False),
    path / "part.parquet")

for p in sorted(OUT.rglob("*.parquet")):
    print(p.relative_to(OUT), f"{p.stat().st_size:,} バイト")

書けました。読むときは**ディレクトリを指すだけ**です。

In [ ]:
back = pd.read_parquet(OUT / "daily_shop")

print(back.shape)
print(back.dtypes)
print()
print(f"合計 {back['amount_jpy'].sum():,}円")

`month` 列が増えているのに注目してください。
**ディレクトリ名から復元されています。** ファイルの中には入っていません。

これがパーティションの利点で、`month=2024-04` のディレクトリを読まなければ、
4月のデータは**1バイトも読み込まれません**。月が100個あっても、
1か月ぶんだけ読むのは一瞬で終わります。

合計は 25,139円。書く前と同じです。

In [ ]:
# ✍ 書いてみる: 書き出した Parquet を読み戻して、店舗別の売上合計を出してください。

ans = ...   # ここに書く

assert ans.to_dict() == {"S01": 8040, "S02": 7340, "S03": 9759}, ans.to_dict()
print("OK")

<details>
<summary>答え</summary>

```python
ans = pd.read_parquet(OUT / "daily_shop").groupby("shop_cd")["amount_jpy"].sum()
```

</details>

---
## 4. 追記すると壊れます

ここからが本題です。**もう一度4月を流したら、どうなるか。**

いちばん素直な書き方は「ファイルを足す」ことです。やってみます。

In [ ]:
# 2回目の実行のつもり。ファイル名を変えて足す
pq.write_table(
    pa.Table.from_pandas(by_shop, schema=SCHEMA_SHOP, preserve_index=False),
    path / "part-2.parquet")

again = pd.read_parquet(OUT / "daily_shop")

print(f"行数   {len(back)} → {len(again)}")
print(f"合計   {back['amount_jpy'].sum():,}円 → {again['amount_jpy'].sum():,}円")

**行数も金額も倍になりました。**

```
30行 25,139円  →  60行 50,278円
```

そして**エラーは1つも出ていません。** 読み物 `docs/01` の言葉でいう「黙る障害」です。

これが冪等でないということです。同じ入力を2回流したのに、結果が変わりました。

> 実務ではもっと巧妙に壊れます。オーケストレータが二重に起動した、
> リトライが走った、誰かが手で再実行した。
> **どれも「2回流れた」だけ**で、この状態になります。
> 気づくのは、たいてい月末に経理から「売上が倍になっている」と連絡が来たときです。

まず片付けます。

In [ ]:
(path / "part-2.parquet").unlink()
print("消した。行数:", len(pd.read_parquet(OUT / "daily_shop")))

In [ ]:
# ✍ 書いてみる: いま daily_shop のパーティションに Parquet ファイルが何個あるか数えてください。

ans = ...   # ここに書く

assert ans == 1, f"1個のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = len(list(partition_path(OUT / "daily_shop", "2024-04").glob("*.parquet")))
```

</details>

---
## 5. 追記ではなく、差し替えにする

冪等にする方法はいくつかあります(読み物 `docs/04` に4つの型が載っています)。
ここで使うのは**型2「パーティション単位の差し替え」**です。実務でいちばんよく使われる形です。

```
その月のパーティションを、まるごと作り直す
```

「足す」のではなく「置き換える」ので、何回やっても同じ状態になります。

### 5-1. 素直に書くと、危ない瞬間があります

```python
shutil.rmtree(part)      # ① 消す
pq.write_table(...)      # ② 書く
```

①と②のあいだで失敗したら、**その月のデータが消えたまま**になります。
処理が5秒かかるなら、5秒間ずっと「4月のデータが存在しない」状態です。

### 5-2. 先に書いてから、最後に入れ替える

順番を変えます。

```
① 一時ディレクトリに、新しい中身を全部書く   ← 失敗してもここで止まるだけ
② 古いパーティションを消す
③ 一時ディレクトリを、パーティションの名前にリネームする   ← 一瞬で終わる
```

**時間のかかる処理を、差し替えの外に出す**のが要点です。

In [ ]:
def write_partition(table_dir, month, df, schema):
    """月ごとのパーティションを、まるごと差し替える。"""
    table_dir = Path(table_dir)
    part = table_dir / f"month={month}"
    tmp = table_dir / f".tmp-{month}"

    # ① 一時ディレクトリに書く
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
    pq.write_table(table, tmp / "part.parquet")

    # ② 古いものを消して ③ 入れ替える
    shutil.rmtree(part, ignore_errors=True)
    os.replace(tmp, part)

    print(f"書いた: {part}  {len(df)}行")


write_partition(OUT / "daily_shop", "2024-04", by_shop, SCHEMA_SHOP)
write_partition(OUT / "daily_category", "2024-04", by_category, SCHEMA_CATEGORY)

`os.replace` はリネームです。**ファイルを1バイトも動かしません**ので、一瞬で終わります。

> **これは完全な原子性ではありません。**
> ②で消してから③でリネームするまでの、ごく短い間だけ、パーティションが存在しません。
> ここまで詰めるには、「どのファイル群が今のテーブルか」をメタデータで管理する仕組みが要ります。
> それがテーブルフォーマット(Iceberg / Delta Lake)で、読み物 `docs/04` に説明があります。
>
> また、S3 のようなオブジェクトストレージには**ディレクトリのリネームがありません。**
> ローカルのファイルシステムだから使える手です。

**それでも、追記よりは桁違いに安全です。** まずこの形にしてください。

In [ ]:
# ✍ 書いてみる: write_partition を使って 4月の daily_shop をもう一度書き、
#              そのあとの行数を ans に入れてください。

ans = ...   # ここに書く

assert ans == 30, f"30行のはずです (増えていたら差し替えになっていません): {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
write_partition(OUT / "daily_shop", "2024-04", by_shop, SCHEMA_SHOP)
ans = len(pd.read_parquet(OUT / "daily_shop"))
```

</details>

**30行のままです。** 4節では倍になったところが、今度は変わりません。

---
## 6. 冪等かどうかを、テストで確かめる

読み物 `docs/04` のチェックリストに、こう書いてあります。

> - [ ] 同じ期間を2回流して、結果が一致することを**テストしたか**

**冪等性は主張ではなく、検証できる性質です。** 実際に2回流して比べます。

In [ ]:
write_partition(OUT / "daily_shop", "2024-04", by_shop, SCHEMA_SHOP)
first = pd.read_parquet(OUT / "daily_shop")

write_partition(OUT / "daily_shop", "2024-04", by_shop, SCHEMA_SHOP)
second = pd.read_parquet(OUT / "daily_shop")

pd.testing.assert_frame_equal(first, second)
print("2回流しても同じでした")
print(f"  {len(first)}行  {first['amount_jpy'].sum():,}円")

`pd.testing.assert_frame_equal` は、**行数・列・型・値・並び順のすべて**を比べます。
1つでも違えば、どこが違うかを教えてくれます。

`==` で比べてはいけません。DataFrame 同士の `==` は
**要素ごとの比較表**を返すだけで、真偽値になりません。

### 冪等性を壊すもの

きれいに書いたつもりでも、次のものが混ざると壊れます。

| 壊すもの | なぜ |
| --- | --- |
| `datetime.date.today()` | **流した日で結果が変わる。** 昨日ぶんを流し直せない |
| 自動採番の連番キー | 流すたびに別の値が振られる |
| 通知や API 呼び出し | 10回流し直すと通知も10回飛ぶ |
| 乱数(シード無し) | 毎回違う結果になる |

いちばん多いのは1行目です。次の節で片付けます。

In [ ]:
# ✍ 書いてみる: daily_category も2回書いて、結果が一致することを確かめてください。
#              一致したら True を ans に入れます。

ans = ...   # ここに書く

assert ans is True
print("OK")

<details>
<summary>答え</summary>

```python
write_partition(OUT / "daily_category", "2024-04", by_category, SCHEMA_CATEGORY)
a = pd.read_parquet(OUT / "daily_category")
write_partition(OUT / "daily_category", "2024-04", by_category, SCHEMA_CATEGORY)
b = pd.read_parquet(OUT / "daily_category")
pd.testing.assert_frame_equal(a, b)
ans = True
```

</details>

---
## 7. 対象期間を、外から渡す

いままで `SOURCES` は4月のファイルを決め打ちしていました。
5月ぶんを流すには、ここを変える必要があります。

**「いつのぶんを流すか」を引数にします。**

```python
run("2024-04")     ← 対象期間を渡す
```

`today()` から求めてはいけません。**昨日ぶんを流し直したいのに、今日ぶんが処理されます。**
これが冪等性を壊すいちばん多い原因です。

### 7-1. ファイルの一覧を、月つきで持つ

In [ ]:
SOURCE_REGISTRY = [
    {"month": "2024-04", "path": "/data/sales_2024-04_old.csv", "encoding": "cp932",
     "tax_type": "税込", "source": "old", "rename": RENAME_STORE},
    {"month": "2024-04", "path": "/data/sales_2024-04_new.csv", "encoding": "utf-8",
     "tax_type": "税込", "source": "new", "rename": RENAME_STORE},
    {"month": "2024-04", "path": "/data/sales_2024-04_ec.csv", "encoding": "utf-8",
     "tax_type": "税抜", "source": "ec", "rename": RENAME_EC},
    {"month": "2024-05", "path": "/data/sales_2024-05_new.csv", "encoding": "utf-8",
     "tax_type": "税込", "source": "new", "rename": RENAME_STORE},
]


def sources_for(month):
    hit = [s for s in SOURCE_REGISTRY if s["month"] == month]
    if not hit:
        raise ValueError(f"{month} のソースが登録されていません")
    return hit


for m in ["2024-04", "2024-05"]:
    print(f"{m}: {[Path(s['path']).name for s in sources_for(m)]}")

01章で作った `SOURCES` に `month` を足しただけです。
**ファイルが増えたら、この表に1行足すだけ**で済むのは01章と同じです。

### 7-2. パイプライン全体を1つの関数にする

In [ ]:
def run(month, out=OUT):
    """指定した月ぶんを、取り込みから出力まで通す。何度呼んでも結果は同じ。"""
    print(f"===== {month} =====")

    raw = pd.concat([read_one(s) for s in sources_for(month)], ignore_index=True)
    print(f"取り込み: {len(raw)}行  内訳 {raw['source'].value_counts().to_dict()}")

    clean, rejected = clean_raw(raw)
    fact = join_master(clean)
    by_shop, by_category = summarize(fact)

    write_partition(out / "daily_shop", month, by_shop, SCHEMA_SHOP)
    write_partition(out / "daily_category", month, by_category, SCHEMA_CATEGORY)
    return by_shop, by_category


_ = run("2024-04")

01章から04章までで作った関数が、**そのまま1本に並んだ**だけです。
新しい処理は1つも足していません。

各章の最後で関数にまとめてきたのは、この形にするためでした。

> `run` の中に `print` がたくさん残っているのに気づいたと思います。
> **これがログです。** 取り込み件数、除外率、未マッチ数、出力行数。
> 毎回の実行でこれが残っていれば、**「いつから様子が変わったか」を後から追えます。**
> 静かに動くパイプラインは、壊れたときに何も手がかりを残しません。

In [ ]:
# ✍ 書いてみる: 登録されていない月を run に渡すと、どうなるか確かめてください。
#              例外の型名を文字列で ans に入れます。

ans = ...   # ここに書く

assert ans == "ValueError", f"'ValueError' のはずです: {ans!r}"
print("OK")

<details>
<summary>答え</summary>

```python
try:
    run("2024-06")
    ans = "例外が出なかった"
except Exception as e:
    ans = type(e).__name__
```

</details>

**知らない月を渡したら、その場で止まります。**

黙って0行のパーティションを書いてしまうと、
「6月は売上ゼロだった」という表ができあがります。
2節のスキーマと同じ考え方で、**分からないときは書かずに止まる**ほうが安全です。

---
## 8. 訂正が届いた (バックフィル)

02章で2行を隔離しました。覚えているでしょうか。

| 日付 | 店 | 商品 | 欠けていたもの |
| --- | --- | --- | --- |
| 4/9 | 横浜店 | 0003 紅茶 | 数量 |
| 4/10 | 渋谷店 | 0001 コーヒー | 金額 |

**その訂正が届きました。** `data/sales_2024-04_fix.csv` です。

In [ ]:
print(open("/data/sales_2024-04_fix.csv", encoding="utf-8").read())

隔離した2行に、欠けていた値が入っています。

**過去の期間を流し直すことを、バックフィルと呼びます。**
冪等な書き込みができていれば、これは**ファイルを1つ足して、もう一度 `run` するだけ**です。

In [ ]:
SOURCE_REGISTRY.append(
    {"month": "2024-04", "path": "/data/sales_2024-04_fix.csv", "encoding": "utf-8",
     "tax_type": "税込", "source": "fix", "rename": RENAME_STORE})

before = pd.read_parquet(OUT / "daily_shop")["amount_jpy"].sum()

by_shop, by_category = run("2024-04")

after = pd.read_parquet(OUT / "daily_shop")["amount_jpy"].sum()
print()
print(f"4月の売上: {before:,}円 → {after:,}円  (+{after - before:,}円)")

**25,139円 → 26,839円。1,700円増えました。**

訂正の2行(900円 + 800円)がそのまま足されています。引き算が合います。

出力の途中経過も見てください。

```
取り込み: 38行            36 + 訂正2行
クレンジング: 38行 → clean 36行 / rejected 2行
```

`rejected` が**2行のまま**なのが大事なところです。
訂正が届いても、**元の壊れた2行は隔離されたまま**です。消えたのは欠損ではなく、
「使える行が2行増えた」というだけです。

そして**パーティションは差し替わったので、行数は30行のまま**です。
追記だったら、ここで60行になって金額も倍になっていました。

> これが「バックフィルのやりやすさは、冪等性がどれだけ設計に入っているかで決まる」
> という読み物 `docs/04` の言葉の意味です。
> 冪等でなければ、ここで手作業の DELETE と INSERT が必要になります。

### 8-1. 訂正でキーが重複していないか確かめる

いまは訂正が「新しい行」として足されました。うまくいったのは、
**元の2行が隔離されていて、集計に入っていなかった**からです。

もし元の行が生きていたら、**同じ売上が2行になります。** 確かめておきます。

In [ ]:
fact_fixed = join_master(clean_raw(
    pd.concat([read_one(s) for s in sources_for("2024-04")], ignore_index=True))[0])

key = ["sale_date", "shop_cd", "item_cd"]
dup = fact_fixed[fact_fixed.duplicated(key, keep=False)].sort_values(key)

print(f"キーが重複している行: {len(dup)}")
print(dup.groupby("source").size().to_dict())

**6行**です。03章の7節で見たのと同じ3組(店頭 + EC)だけで、
**`fix` は1行も入っていません。** 訂正は既存の行とぶつかっていません。

もしぶつかっていたら、どちらを採るかを決める必要がありました。

```python
# 「あとから来たほうが正しい」と決めるなら
fact.sort_values("source").drop_duplicates(key, keep="last")
```

読み物 `docs/03` の言うとおり、そのためには**どれが最新かを決める列**が要ります
(`ingested_at` のような取り込み時刻)。このデータにはそれがないので、
**そもそもぶつからない設計**にしてあります。

> 順番にも決まりがあります。**「選んでから、捨てる」**です。
> 重複排除より先に除外をすると、訂正で無効になった古い行が生き残ります。
> 読み物 `docs/03` の「7と8の順番」に、例つきで書いてあります。

In [ ]:
# ✍ 書いてみる: 訂正後の4月の店舗別売上を、Parquetから読んで出してください。

ans = ...   # ここに書く

assert ans.to_dict() == {"S01": 8940, "S02": 7340, "S03": 10559}, ans.to_dict()
print("OK")
print(ans)

<details>
<summary>答え</summary>

```python
ans = pd.read_parquet(OUT / "daily_shop").groupby("shop_cd")["amount_jpy"].sum()
```

</details>

渋谷店が 8,040 → 8,940(+900)、横浜店が 9,759 → 10,559(+800)。
**訂正の行き先も、狙いどおりです。**

---
## 9. 5月を流す (増分)

新しい月のデータが届きました。**4月には触らずに、5月だけ足します。**

In [ ]:
apr_before = pd.read_parquet(
    OUT / "daily_shop" / "month=2024-04")["amount_jpy"].sum()

_ = run("2024-05")

apr_after = pd.read_parquet(
    OUT / "daily_shop" / "month=2024-04")["amount_jpy"].sum()

print()
print(f"4月: {apr_before:,}円 → {apr_after:,}円  (触っていない)")
assert apr_before == apr_after

**4月は1円も変わっていません。** `write_partition` が触るのは
`month=2024-05` のディレクトリだけだからです。

これがパーティションを月で切った理由です。
**流し直す単位と、置き場所の単位を合わせておく**と、他の月に影響が出ません。

出力を見ると、5月は行数が違います。

```
サマリ: 店舗別 24行 (うち売上ゼロ 16行) / カテゴリ別 8行
```

5/1〜5/8 の8日ぶん × 3店 = 24行。うち16行がゼロです。
**04章で決めた「データのある範囲だけ埋める」が効いています。**
5月末まで埋めていたら 31日 × 3 = 93行になり、大半が意味の無いゼロになっていました。

In [ ]:
whole = pd.read_parquet(OUT / "daily_shop")

print(whole.groupby("month")["amount_jpy"].agg(["count", "sum"]))
print()
print(f"全期間の合計: {whole['amount_jpy'].sum():,}円")

`out/daily_shop` をディレクトリごと読むと、**2か月ぶんが1枚の表**になります。

```
month
2024-04   30行  26,839円
2024-05   24行   6,780円
```

書くときは月ごと、読むときはまとめて。**これがパーティションの使い方**です。

In [ ]:
# ✍ 書いてみる: 5月だけを読んで、いちばん売れた店の shop_cd を出してください。

ans = ...   # ここに書く

assert ans == "S03", f"'S03' のはずです: {ans!r}"
print("OK")

<details>
<summary>答え</summary>

```python
may = pd.read_parquet(OUT / "daily_shop" / "month=2024-05")
ans = may.groupby("shop_cd")["amount_jpy"].sum().idxmax()
```

</details>

---
## 10. 全部まとめて、もう一度流す

最後に、**何もかも最初からやり直します。** 出力を全部消して、両方の月を流し直します。

In [ ]:
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)

for month in ["2024-04", "2024-05"]:
    run(month)
    print()

for p in sorted(OUT.rglob("*.parquet")):
    print(p.relative_to(OUT))

出力を全部消してから流し直しても、**同じものができあがります。**

これが冪等なパイプラインの姿です。
「途中まで流れた状態」を気にせずに、いつでも頭から流せます。

### 最後の確認

In [ ]:
final = pd.read_parquet(OUT / "daily_shop")

print(final.groupby("month")["amount_jpy"].sum().to_dict())

assert final.groupby("month")["amount_jpy"].sum().to_dict() == {
    "2024-04": 26839, "2024-05": 6780}
assert final["sale_date"].map(type).eq(__import__("datetime").date).all(), "日付型が保たれていること"
print("\nOK")

```
2024-04  26,839円
2024-05   6,780円
```

生ログ3種+訂正から、ここまで来ました。

```
sales_2024-04_old.csv   cp932・全角・和風日付      14行
sales_2024-04_new.csv   列順違い・￥とカンマ・N/A   12行
sales_2024-04_ec.csv    税抜・返品・先頭ゼロ落ち    10行
sales_2024-04_fix.csv   訂正                       2行
sales_2024-05_new.csv   5月分                      8行
        ↓
out/daily_shop/month=2024-04/part.parquet      30行
out/daily_shop/month=2024-05/part.parquet      24行
out/daily_category/month=2024-04/part.parquet  29行
out/daily_category/month=2024-05/part.parquet   8行
```

In [ ]:
# ✍ 書いてみる: run を書いた順と逆に、5月 → 4月 の順で流しても
#              結果が同じになることを確かめてください。同じなら True を ans に入れます。

ans = ...   # ここに書く

assert ans is True
print("OK")

<details>
<summary>答え</summary>

```python
before = pd.read_parquet(OUT / "daily_shop")
for month in ["2024-05", "2024-04"]:
    run(month)
after = pd.read_parquet(OUT / "daily_shop")
pd.testing.assert_frame_equal(
    before.sort_values(["month", "sale_date", "shop_cd"]).reset_index(drop=True),
    after.sort_values(["month", "sale_date", "shop_cd"]).reset_index(drop=True))
ans = True
```

</details>

**流す順番を変えても同じです。** 月ごとに独立しているので、
どの順で流しても、何回流しても、同じ結果になります。

並べて流すことも、失敗した月だけ流し直すこともできます。
**これがバックフィルのやりやすさ**です。

---
## この章で分かったこと

| | |
| --- | --- |
| Parquet | 型を持って保存できる。CSV は型を持てず、下流に推測をやり直させる |
| スキーマ | 出口で型を強制する。**黙る障害を、止まる障害に変える** |
| パーティション | **分ける単位は「流し直す単位」**で決める |
| 追記 | 2回流すと倍になる。エラーは出ない |
| 差し替え | 一時ディレクトリに書いて、最後にリネーム。**時間のかかる処理を差し替えの外に出す** |
| 冪等の検証 | 主張ではなく**テストする**。`assert_frame_equal` で2回ぶんを比べる |
| 対象期間 | **引数で受け取る。`today()` を使わない** |
| バックフィル | 冪等なら、ファイルを足してもう一度流すだけ |
| 増分 | 他の月のパーティションに触らない |
| ログ | 件数・除外率・未マッチ数を毎回出す。**変化に気づけるのは記録があるときだけ** |

## この教材で作ったもの

5章かけて作ったのは、この4つの関数です。

```python
raw   = load_raw(sources)          # 01章  文字コード・列順の違いを吸収して1枚にする
clean, rejected = clean_raw(raw)   # 02章  表記・欠損・型・税を整え、使えない行を隔離する
fact  = join_master(clean)         # 03章  名寄せしてマスタを結合する。行は増やさない
by_shop, by_category = summarize(fact)   # 04章  除外を数えて集計する
                                   # 05章  スキーマを強制して、差し替えで書く
```

**これが取り込み層(staging / bronze)の基本の形です。**
題材が POS でも注文でもログでも、やることの並びは変わりません。

読み物 `docs/03` の言葉でいうと、この5章がやってきたのは1つのことです。

> **下流が上流の事情を知らなくて済むようにする。**

`out/daily_shop` を読む人は、cp932 も全角数字も税抜も知りません。知る必要がありません。
それを全部この5章で引き受けました。

## 次にやること

**腕試しは `quest-01 raw-ingest` です。** 今度は仕様書だけ渡されて、自分で実装します。

```bash
cd quest-01-raw-ingest
cat README.md          # ミッション確認
./setup.sh             # 環境と生データを用意する
cat spec/orders.md     # 仕様を読む
```

この教材との違いはこうです。

| | tutorial-01 | quest-01 |
| --- | --- | --- |
| 手順 | ノートブックに書いてある | **仕様書だけ**(`spec/orders.md`) |
| 答え | 各問に付いている | **無い** |
| データ | 毎回同じ36行 | **毎回作り直される**(行数も合計も変わる) |
| 判定 | `assert` が通れば OK | 見張り役が検証し、通っている間だけ FLAG が出る |

**やることは同じです。** 読む → 整える → 重複排除 → 除外 → 型を固めて書く。
この教材でやったことを、順番に思い出しながら書けば通ります。

詰まったら、`docs/appendix 「やりたいことが書けない」を抜ける` を読んでください。
手の動かし方の話が書いてあります。